# Stage 1E: QA Generation with Gemini API

| Version | Date | Author | Description |
| --- | --- | --- | --- |
| 2.0.0 | 2026-01-28 | That Le | Incremental QA generation with skip existing |

---

## Purpose

Generate QA pairs for classified charts using Gemini API:
- **Load existing QA** from `dataset.json` (2,852 images, 13,297 QA pairs)
- **Skip already processed** images to save API cost
- **Generate QA** for new charts only

### Pipeline Position

```
01d: Classification --> classified_charts/
        |
[01e: QA Generation]  <-- This notebook (Gemini API)
        |
        v
chart_qa/dataset.json  --> Stage 4 (SLM Training)
```

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

EXECUTE_GENERATION = False  # Set True to generate (uses API credits)
BATCH_SIZE = 50             # Images per batch before checkpoint
RATE_LIMIT_SECONDS = 1.0    # Delay between API calls
MAX_NEW_IMAGES = None       # None = all pending, or set limit

# Gemini model
GEMINI_MODEL = "gemini-3-flash-preview"  # or "gemini-1.5-flash-preview"

print(f"Mode: {'ACTIVE' if EXECUTE_GENERATION else 'DRY RUN'}")
print(f"Model: {GEMINI_MODEL}")

In [ ]:
# ============================================================================
# SETUP
# ============================================================================
import sys
import os
from pathlib import Path
from datetime import datetime
import json
import time

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load .env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# Directories
CLASSIFIED_DIR = PROJECT_ROOT / "data" / "academic_dataset" / "classified_charts"
QA_DIR = PROJECT_ROOT / "data" / "academic_dataset" / "chart_qa"
DATASET_FILE = QA_DIR / "dataset.json"
CHECKPOINT_FILE = QA_DIR / "generation_checkpoint.json"

QA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Classified charts: {CLASSIFIED_DIR}")
print(f"QA output: {QA_DIR}")

In [ ]:
# ============================================================================
# LOAD EXISTING DATA
# ============================================================================

def load_existing_qa():
    """Load existing QA dataset."""
    if DATASET_FILE.exists():
        with open(DATASET_FILE) as f:
            data = json.load(f)
        # Extract processed image IDs
        processed_ids = set()
        samples = data.get("samples", [])
        for s in samples:
            processed_ids.add(s.get("image_id"))
        return data, processed_ids
    return None, set()


existing_data, processed_ids = load_existing_qa()

print("=" * 60)
print("EXISTING QA DATA")
print("=" * 60)
if existing_data:
    print(f"Total images:   {existing_data.get('total_images', 0):,}")
    print(f"Total QA pairs: {existing_data.get('total_qa_pairs', 0):,}")
    print(f"Created at:     {existing_data.get('created_at', 'N/A')}")
else:
    print("No existing data found.")
print("=" * 60)

In [ ]:
# ============================================================================
# FIND PENDING IMAGES
# ============================================================================

def get_pending_images():
    """Get images that haven't been processed yet."""
    pending = []
    
    # Scan classified charts (skip uncertain)
    for type_dir in CLASSIFIED_DIR.iterdir():
        if type_dir.is_dir() and type_dir.name != "uncertain":
            chart_type = type_dir.name
            for img_path in type_dir.glob("*.png"):
                image_id = img_path.stem
                if image_id not in processed_ids:
                    pending.append({
                        "image_id": image_id,
                        "image_path": str(img_path),
                        "chart_type": chart_type,
                    })
    
    return pending


pending_images = get_pending_images()

# Count by type
by_type = {}
for img in pending_images:
    t = img["chart_type"]
    by_type[t] = by_type.get(t, 0) + 1

print("=" * 60)
print("PENDING IMAGES (not yet processed)")
print("=" * 60)
print(f"Total pending: {len(pending_images):,}")
print(f"Already done:  {len(processed_ids):,}")
print("-" * 60)
print("By type:")
for t, c in sorted(by_type.items(), key=lambda x: -x[1]):
    print(f"  {t:12s}: {c:,}")
print("=" * 60)

In [ ]:
# ============================================================================
# GEMINI API SETUP
# ============================================================================
import google.generativeai as genai
from PIL import Image
import base64
import io

# Configure API
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    print("[WARNING] No API key found. Set GEMINI_API_KEY in .env")
else:
    genai.configure(api_key=api_key)
    print(f"Gemini API configured")
    print(f"Model: {GEMINI_MODEL}")

In [ ]:
# ============================================================================
# QA GENERATION PROMPT
# ============================================================================

QA_PROMPT = """
Analyze this {chart_type} chart and generate 5 question-answer pairs.

Generate questions in these categories:
1. STRUCTURAL: About title, labels, legend, axes
2. COUNTING: Count elements (bars, lines, slices, etc.)
3. COMPARISON: Compare values or categories
4. EXTRACTION: Extract specific values
5. REASONING: Trends, patterns, insights

Respond in this JSON format:
{{
  "chart_description": "Brief description of the chart",
  "qa_pairs": [
    {{
      "category": "structural",
      "question": "What is the title of this chart?",
      "answer": "The title is..."
    }},
    ...
  ]
}}

Rules:
- Questions must be answerable from the chart image alone
- Answers must be accurate based on visible data
- Keep answers concise but complete
- If something is unclear, note it in the answer
"""

print("QA prompt template ready.")

In [ ]:
# ============================================================================
# GENERATION FUNCTION
# ============================================================================

def generate_qa_for_image(image_path: str, chart_type: str, model_name: str = GEMINI_MODEL):
    """
    Generate QA pairs for a single image using Gemini.
    
    Returns:
        dict with qa_pairs or None on error
    """
    try:
        # Load image
        img = Image.open(image_path)
        
        # Create model
        model = genai.GenerativeModel(model_name)
        
        # Generate
        prompt = QA_PROMPT.format(chart_type=chart_type)
        response = model.generate_content([prompt, img])
        
        # Parse JSON from response
        text = response.text
        
        # Extract JSON (handle markdown code blocks)
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        elif "```" in text:
            text = text.split("```")[1].split("```")[0]
        
        result = json.loads(text.strip())
        return result
        
    except Exception as e:
        print(f"Error: {e}")
        return None


# Test on one image
if pending_images and api_key:
    test_img = pending_images[0]
    print(f"Testing on: {test_img['image_id']}")
    print(f"Type: {test_img['chart_type']}")
    
    if EXECUTE_GENERATION:
        result = generate_qa_for_image(test_img["image_path"], test_img["chart_type"])
        if result:
            print(f"Generated {len(result.get('qa_pairs', []))} QA pairs")
            print(json.dumps(result, indent=2)[:500])
    else:
        print("[DRY RUN] Set EXECUTE_GENERATION = True to test")

In [ ]:
# ============================================================================
# BATCH GENERATION WITH CHECKPOINTS
# ============================================================================
from tqdm.auto import tqdm


def load_checkpoint():
    """Load generation checkpoint."""
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {"processed": [], "results": [], "errors": []}


def save_checkpoint(checkpoint):
    """Save generation checkpoint."""
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=2)


def batch_generate(
    images: list,
    batch_size: int = 50,
    rate_limit: float = 1.0,
    max_images: int = None,
):
    """
    Generate QA for multiple images with checkpointing.
    """
    checkpoint = load_checkpoint()
    checkpoint_ids = set(checkpoint["processed"])
    
    # Filter already in checkpoint
    pending = [img for img in images if img["image_id"] not in checkpoint_ids]
    
    if max_images:
        pending = pending[:max_images]
    
    print("=" * 60)
    print(f"BATCH GENERATION")
    print(f"Pending: {len(pending):,} images")
    print(f"Batch size: {batch_size}")
    print(f"Rate limit: {rate_limit}s between calls")
    print("=" * 60)
    
    new_results = []
    
    for i, img in enumerate(tqdm(pending, desc="Generating QA")):
        try:
            result = generate_qa_for_image(img["image_path"], img["chart_type"])
            
            if result:
                entry = {
                    "image_id": img["image_id"],
                    "image_path": img["image_path"],
                    "chart_type": img["chart_type"],
                    "description": result.get("chart_description", ""),
                    "qa_pairs": result.get("qa_pairs", []),
                }
                new_results.append(entry)
                checkpoint["results"].append(entry)
            else:
                checkpoint["errors"].append(img["image_id"])
            
            checkpoint["processed"].append(img["image_id"])
            
            # Checkpoint every batch
            if (i + 1) % batch_size == 0:
                save_checkpoint(checkpoint)
                print(f"  Checkpoint: {i + 1}/{len(pending)}")
            
            # Rate limit
            time.sleep(rate_limit)
            
        except KeyboardInterrupt:
            print("\n[INTERRUPTED] Saving checkpoint...")
            save_checkpoint(checkpoint)
            break
        except Exception as e:
            print(f"Error on {img['image_id']}: {e}")
            checkpoint["errors"].append(img["image_id"])
    
    # Final save
    save_checkpoint(checkpoint)
    
    print("\n" + "=" * 60)
    print(f"GENERATION COMPLETE")
    print(f"New QA generated: {len(new_results):,}")
    print(f"Errors: {len(checkpoint['errors']):,}")
    print("=" * 60)
    
    return new_results

In [ ]:
# ============================================================================
# EXECUTE GENERATION
# ============================================================================

if EXECUTE_GENERATION and pending_images and api_key:
    new_results = batch_generate(
        images=pending_images,
        batch_size=BATCH_SIZE,
        rate_limit=RATE_LIMIT_SECONDS,
        max_images=MAX_NEW_IMAGES,
    )
else:
    new_results = []
    print("[SKIPPED] Generation not executed")
    print(f"  EXECUTE_GENERATION = {EXECUTE_GENERATION}")
    print(f"  Pending images: {len(pending_images):,}")
    print(f"  API key set: {bool(api_key)}")

In [ ]:
# ============================================================================
# MERGE WITH EXISTING DATA
# ============================================================================

def merge_and_save(existing_data, new_results, output_file):
    """
    Merge new results with existing dataset.
    """
    if existing_data is None:
        existing_data = {
            "version": "2.0.0",
            "created_at": datetime.now().isoformat(),
            "total_images": 0,
            "total_qa_pairs": 0,
            "samples": [],
        }
    
    # Add new results
    existing_ids = {s["image_id"] for s in existing_data["samples"]}
    
    added = 0
    new_qa = 0
    for result in new_results:
        if result["image_id"] not in existing_ids:
            existing_data["samples"].append(result)
            added += 1
            new_qa += len(result.get("qa_pairs", []))
    
    # Update totals
    existing_data["total_images"] = len(existing_data["samples"])
    existing_data["total_qa_pairs"] = sum(
        len(s.get("qa_pairs", [])) for s in existing_data["samples"]
    )
    existing_data["updated_at"] = datetime.now().isoformat()
    
    # Save
    with open(output_file, "w") as f:
        json.dump(existing_data, f, indent=2)
    
    print(f"Added {added:,} images with {new_qa:,} QA pairs")
    print(f"Total: {existing_data['total_images']:,} images, {existing_data['total_qa_pairs']:,} QA pairs")
    print(f"Saved to: {output_file}")
    
    return existing_data


if new_results:
    updated_data = merge_and_save(existing_data, new_results, DATASET_FILE)
else:
    print("No new results to merge.")

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================

# Reload final data
if DATASET_FILE.exists():
    with open(DATASET_FILE) as f:
        final_data = json.load(f)
    
    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Version:        {final_data.get('version', 'N/A')}")
    print(f"Total images:   {final_data.get('total_images', 0):,}")
    print(f"Total QA pairs: {final_data.get('total_qa_pairs', 0):,}")
    print(f"Avg QA/image:   {final_data.get('total_qa_pairs', 0) / max(final_data.get('total_images', 1), 1):.1f}")
    print("-" * 60)
    
    # By chart type
    type_counts = {}
    for s in final_data.get("samples", []):
        t = s.get("chart_type", "unknown")
        type_counts[t] = type_counts.get(t, 0) + 1
    
    print("By chart type:")
    for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"  {t:12s}: {c:,}")
    print("=" * 60)

---

## Summary

This notebook:
1. **Loads existing QA** from `dataset.json`
2. **Finds pending** images not yet processed
3. **Generates QA** using Gemini API
4. **Saves incrementally** with checkpoints
5. **Merges** new data with existing

### Cost Optimization

- Skip already processed images
- Checkpoint every 50 images
- Resume from checkpoint on interrupt

### Next: Stage 4 (SLM)

Use the generated QA dataset to:
- Fine-tune SLM for chart reasoning
- Train question-answering model